<a href="https://colab.research.google.com/github/lis-r-barreto/llm-study-notes/blob/main/hugging-face-llm-course/Chapter_2_Section_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Behind the pipeline (PyTorch)

This notebook demonstrates how to use the Hugging Face `transformers` library, focusing on sentiment analysis. We'll explore the `pipeline` API and then dive deeper into the underlying components like tokenizers and models.

In [ ]:
!pip install datasets evaluate transformers[sentencepiece]

## 1. Using the `pipeline` API

The `pipeline` API is the easiest way to use pre-trained models for various tasks. Here, we're using the `sentiment-analysis` pipeline, which can classify text as positive or negative.

In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
classifier(
    [
        "I've been waiting for a HuggingFace course my whole life.",
        "I hate this so much!",
    ]
)


## 2. Behind the `pipeline`: Tokenization

Under the hood, the `pipeline` breaks down the text into numerical representations that a model can understand. This process is called tokenization. We use `AutoTokenizer` to load the tokenizer corresponding to our pre-trained model checkpoint.

In [ ]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

The `tokenizer` converts raw text into input IDs, attention masks, and token type IDs. `padding=True` ensures all sequences have the same length, `truncation=True` cuts off longer sequences, and `return_tensors='pt'` returns PyTorch tensors.

In [ ]:
raw_inputs = [
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!",
]
inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt")
print(inputs)

## 3. Behind the `pipeline`: Model Loading and Output

Next, we load the pre-trained model. `AutoModel` loads the base model without a specific head for a task. Its output is typically the hidden states (numerical representations) of the input.

In [ ]:
from transformers import AutoModel

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModel.from_pretrained(checkpoint)


When the `inputs` are passed to the model, it produces an output. For `AutoModel`, the `last_hidden_state` contains the contextualized embeddings for each token in the input sequences. The shape indicates `(batch_size, sequence_length, hidden_size)`.

In [ ]:
outputs = model(**inputs)
print(outputs.last_hidden_state.shape)

## 4. Task-Specific Models: `AutoModelForSequenceClassification`

For specific tasks like sentiment analysis, there are task-specific models (e.g., `AutoModelForSequenceClassification`) that include a classification head on top of the base model. This head is designed to produce logits for classification.

In [ ]:
from transformers import AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
outputs = model(**inputs)

The `logits` are the raw, unnormalized scores from the model's classification head. Their shape `(batch_size, num_labels)` indicates one logit for each class (e.g., positive/negative) per input sentence.

In [ ]:
print(outputs.logits.shape)

In [ ]:
print(outputs.logits)

## 5. Converting Logits to Probabilities

To get actual probabilities from the logits, we apply a `softmax` function. The `softmax` function converts a vector of numbers into a probability distribution, where each value is between 0 and 1 and the sum of all values is 1.

In [ ]:
import torch

predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
print(predictions)

## 6. Interpreting Predictions: `id2label`

The `model.config.id2label` attribute provides a mapping from the numerical IDs of the output classes (e.g., 0, 1) to their human-readable labels (e.g., 'NEGATIVE', 'POSITIVE'). This helps us interpret what each probability score corresponds to.

In [ ]:
model.config.id2label

## 7. Try it out! (Replication Example)

This section demonstrates how to apply the same steps shown above with new input texts to verify the understanding of the `pipeline`'s inner workings.

✏️ Try it out! Choose two (or more) texts of your own and run them through the sentiment-analysis pipeline. Then replicate the steps you saw here yourself and check that you obtain the same results!

In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
classifier(
    [
        "I love David Lynch movies.",
        "I hate Adam Sandler movies!",
    ]
)

In [ ]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [ ]:
raw_inputs = [
    "I love David Lynch movies.",
    "I hate papaya!",
]
inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt")
print(inputs)

In [ ]:
from transformers import AutoModel

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModel.from_pretrained(checkpoint)

In [ ]:
outputs = model(**inputs)
print(outputs.last_hidden_state.shape)

In [ ]:
from transformers import AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
outputs = model(**inputs)

In [ ]:
print(outputs.logits.shape)

In [ ]:
print(outputs.logits)

In [ ]:
import torch

predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
print(predictions)

In [ ]:
model.config.id2label

## Resources

- [Hugging Face Transformers Documentation](https://huggingface.co/docs/transformers/index)
- [Sentiment Analysis with Transformers](https://huggingface.co/tasks/sentiment-analysis)